## 1. Setup and Installation

In [ ]:
# Install required packages
%pip install numpy pandas matplotlib seaborn scikit-learn tensorflow joblib -q

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.utils import to_categorical
import os
import json
import joblib
from datetime import datetime
from copy import deepcopy

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

## 2. Mount Google Drive and Load Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Set dataset path
DATASET_PATH = '/content/drive/MyDrive/UCI_HAR_Dataset'

# Verify path
if os.path.exists(DATASET_PATH):
    print(f"✅ Dataset found at: {DATASET_PATH}")
else:
    print(f"❌ Dataset not found. Please upload UCI_HAR_Dataset to Google Drive.")

## 3. Load and Prepare Data

In [ ]:
def load_data(dataset_path):
    """
    Load UCI HAR dataset.
    """
    # Load training data
    X_train = np.loadtxt(os.path.join(dataset_path, 'train', 'X_train.txt'))
    y_train = np.loadtxt(os.path.join(dataset_path, 'train', 'y_train.txt'))
    subject_train = np.loadtxt(os.path.join(dataset_path, 'train', 'subject_train.txt'))
    
    # Load test data
    X_test = np.loadtxt(os.path.join(dataset_path, 'test', 'X_test.txt'))
    y_test = np.loadtxt(os.path.join(dataset_path, 'test', 'y_test.txt'))
    subject_test = np.loadtxt(os.path.join(dataset_path, 'test', 'subject_test.txt'))
    
    # Load activity labels
    with open(os.path.join(dataset_path, 'activity_labels.txt'), 'r') as f:
        activity_labels = [line.strip().split()[1] for line in f.readlines()]
    
    return X_train, y_train, subject_train, X_test, y_test, subject_test, activity_labels

# Load data
print("Loading dataset...")
X_train, y_train, subject_train, X_test, y_test, subject_test, activity_labels = load_data(DATASET_PATH)

print(f"\nDataset loaded successfully!")
print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")
print(f"Features: {X_train.shape[1]}")
print(f"Activities: {activity_labels}")

In [ ]:
# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert labels to categorical (subtract 1 for 0-indexing)
y_train_cat = to_categorical(y_train - 1, num_classes=6)
y_test_cat = to_categorical(y_test - 1, num_classes=6)

print("Data preprocessing completed!")

## 4. Define Shared Model Architecture

**Important:** Both FL and DL use the exact same architecture for fair comparison.

In [ ]:
def create_model(input_shape=(561,), num_classes=6):
    """
    Create model with identical architecture for both FL and DL.
    
    Architecture:
    - Input: 561 features
    - Hidden: 512 -> 256 -> 128 -> 64 (with BatchNorm and Dropout)
    - Output: 6 classes (softmax)
    """
    model = models.Sequential([
        # Input layer
        layers.Input(shape=input_shape),
        
        # First hidden layer
        layers.Dense(512, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        
        # Second hidden layer
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.4),
        
        # Third hidden layer
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        
        # Fourth hidden layer
        layers.Dense(64, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.2),
        
        # Output layer
        layers.Dense(num_classes, activation='softmax')
    ])
    
    # Compile model
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

# Display model architecture
test_model = create_model()
test_model.summary()

## 5. Deep Learning Training Function

In [ ]:
def train_dl_model(X_train, y_train, X_test, y_test, epochs=20, batch_size=32, model_name="dl_model"):
    """
    Train Deep Learning model (centralized).
    
    Args:
        X_train: Training features
        y_train: Training labels (categorical)
        X_test: Test features
        y_test: Test labels (categorical)
        epochs: Number of epochs to train
        batch_size: Batch size
        model_name: Name for saving model
    
    Returns:
        model, history, test_accuracy
    """
    print(f"\n{'='*60}")
    print(f"Training DL Model - {epochs} Epochs")
    print(f"{'='*60}\n")
    
    # Create model
    model = create_model()
    
    # Train model
    history = model.fit(
        X_train,
        y_train,
        batch_size=batch_size,
        epochs=epochs,
        validation_split=0.2,
        verbose=1
    )
    
    # Evaluate on test set
    test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
    
    print(f"\n✅ DL Model Training Complete!")
    print(f"Test Accuracy: {test_accuracy*100:.2f}%")
    
    # Save model
    model.save(f"{model_name}_epochs{epochs}.h5")
    
    return model, history, test_accuracy

## 6. Federated Learning Training Function

In [ ]:
def prepare_federated_data(X_train, y_train, subject_train):
    """
    Prepare data for federated learning by splitting by subjects.
    """
    unique_subjects = np.unique(subject_train)
    client_data = {}
    
    for subject_id in unique_subjects:
        mask = subject_train == subject_id
        client_data[int(subject_id)] = {
            'X': X_train[mask],
            'y': y_train[mask],
            'num_samples': np.sum(mask)
        }
    
    return client_data

def train_client_model(global_weights, client_data, epochs=5, batch_size=32):
    """
    Train a local model on client data.
    """
    # Create local model
    local_model = create_model()
    
    # Set global weights
    local_model.set_weights(global_weights)
    
    # Train locally
    history = local_model.fit(
        client_data['X'],
        client_data['y'],
        batch_size=batch_size,
        epochs=epochs,
        verbose=0,
        validation_split=0.0
    )
    
    return local_model.get_weights(), history

def federated_averaging(client_weights_list, client_sample_counts):
    """
    Perform federated averaging (FedAvg).
    """
    total_samples = sum(client_sample_counts)
    averaged_weights = []
    
    for layer_idx in range(len(client_weights_list[0])):
        layer_avg = np.zeros_like(client_weights_list[0][layer_idx])
        
        for client_idx, client_weights in enumerate(client_weights_list):
            weight = client_sample_counts[client_idx] / total_samples
            layer_avg += weight * client_weights[layer_idx]
        
        averaged_weights.append(layer_avg)
    
    return averaged_weights

def train_fl_model(X_train, y_train, subject_train, X_test, y_test, num_rounds=20, 
                   local_epochs=5, batch_size=32, clients_per_round=10, model_name="fl_model"):
    """
    Train Federated Learning model.
    
    Args:
        X_train: Training features
        y_train: Training labels (categorical)
        subject_train: Subject IDs for splitting
        X_test: Test features
        y_test: Test labels (categorical)
        num_rounds: Number of federated rounds
        local_epochs: Epochs per client per round
        batch_size: Batch size
        clients_per_round: Clients sampled per round
        model_name: Name for saving model
    
    Returns:
        global_model, fl_history, test_accuracy
    """
    print(f"\n{'='*60}")
    print(f"Training FL Model - {num_rounds} Rounds")
    print(f"{'='*60}\n")
    
    # Prepare client data
    client_data = prepare_federated_data(X_train, y_train, subject_train)
    client_ids = list(client_data.keys())
    
    print(f"Total clients: {len(client_ids)}")
    print(f"Clients per round: {clients_per_round}")
    print(f"Local epochs: {local_epochs}\n")
    
    # Create global model
    global_model = create_model()
    
    # Track history
    fl_history = {
        'round': [],
        'test_accuracy': [],
        'test_loss': []
    }
    
    # Federated training loop
    for round_num in range(1, num_rounds + 1):
        print(f"Round {round_num}/{num_rounds}", end="")
        
        # Sample clients
        sampled_clients = np.random.choice(client_ids, size=clients_per_round, replace=False)
        
        # Get global weights
        global_weights = global_model.get_weights()
        
        # Train on each client
        client_weights_list = []
        client_sample_counts = []
        
        for client_id in sampled_clients:
            client_weights, _ = train_client_model(
                global_weights,
                client_data[client_id],
                epochs=local_epochs,
                batch_size=batch_size
            )
            client_weights_list.append(client_weights)
            client_sample_counts.append(client_data[client_id]['num_samples'])
        
        # Aggregate weights
        averaged_weights = federated_averaging(client_weights_list, client_sample_counts)
        global_model.set_weights(averaged_weights)
        
        print(" - Complete")
    
    # Evaluate ONLY at the end (like DL does)
    print("\nEvaluating final model on test set...")
    test_loss, test_accuracy = global_model.evaluate(X_test, y_test, verbose=0)
    
    # Store only final result
    fl_history['round'].append(num_rounds)
    fl_history['test_accuracy'].append(test_accuracy)
    fl_history['test_loss'].append(test_loss)
    
    print(f"\n✅ FL Model Training Complete!")
    print(f"Final Test Accuracy: {test_accuracy*100:.2f}%")
    
    # Save model
    global_model.save(f"{model_name}_rounds{num_rounds}.h5")
    
    return global_model, fl_history, test_accuracy

## 7. Train All Models with Different Configurations

Training 6 models total:
- **DL 20 epochs**, FL 20 rounds
- **DL 10 epochs**, FL 10 rounds
- **DL 5 epochs**, FL 5 rounds

In [ ]:
# Storage for results
results = {
    'dl_20': None,
    'fl_20': None,
    'dl_10': None,
    'fl_10': None,
    'dl_5': None,
    'fl_5': None
}

### 7.1 Train with 20 Rounds/Epochs

In [ ]:
# Train DL model - 20 epochs
dl_model_20, dl_history_20, dl_acc_20 = train_dl_model(
    X_train_scaled, y_train_cat, X_test_scaled, y_test_cat,
    epochs=20, batch_size=32, model_name="dl_model"
)
results['dl_20'] = {'model': dl_model_20, 'history': dl_history_20, 'accuracy': dl_acc_20}

In [ ]:
# Train FL model - 20 rounds (fair comparison: 1 client per round, 1 local epoch)
fl_model_20, fl_history_20, fl_acc_20 = train_fl_model(
    X_train_scaled, y_train_cat, subject_train, X_test_scaled, y_test_cat,
    num_rounds=20, local_epochs=1, batch_size=32, clients_per_round=1, model_name="fl_model"
)
results['fl_20'] = {'model': fl_model_20, 'history': fl_history_20, 'accuracy': fl_acc_20}

### 7.2 Train with 10 Rounds/Epochs

In [ ]:
# Train DL model - 10 epochs
dl_model_10, dl_history_10, dl_acc_10 = train_dl_model(
    X_train_scaled, y_train_cat, X_test_scaled, y_test_cat,
    epochs=10, batch_size=32, model_name="dl_model"
)
results['dl_10'] = {'model': dl_model_10, 'history': dl_history_10, 'accuracy': dl_acc_10}

In [ ]:
# Train FL model - 10 rounds (fair comparison: 1 client per round, 1 local epoch)
fl_model_10, fl_history_10, fl_acc_10 = train_fl_model(
    X_train_scaled, y_train_cat, subject_train, X_test_scaled, y_test_cat,
    num_rounds=10, local_epochs=1, batch_size=32, clients_per_round=1, model_name="fl_model"
)
results['fl_10'] = {'model': fl_model_10, 'history': fl_history_10, 'accuracy': fl_acc_10}

### 7.3 Train with 5 Rounds/Epochs

In [ ]:
# Train DL model - 5 epochs
dl_model_5, dl_history_5, dl_acc_5 = train_dl_model(
    X_train_scaled, y_train_cat, X_test_scaled, y_test_cat,
    epochs=5, batch_size=32, model_name="dl_model"
)
results['dl_5'] = {'model': dl_model_5, 'history': dl_history_5, 'accuracy': dl_acc_5}

In [ ]:
# Train FL model - 5 rounds (fair comparison: 1 client per round, 1 local epoch)
fl_model_5, fl_history_5, fl_acc_5 = train_fl_model(
    X_train_scaled, y_train_cat, subject_train, X_test_scaled, y_test_cat,
    num_rounds=5, local_epochs=1, batch_size=32, clients_per_round=1, model_name="fl_model"
)
results['fl_5'] = {'model': fl_model_5, 'history': fl_history_5, 'accuracy': fl_acc_5}

## 8. Compare Results

In [ ]:
# Create comparison summary
comparison_df = pd.DataFrame({
    'Configuration': ['20 Rounds/Epochs', '10 Rounds/Epochs', '5 Rounds/Epochs'],
    'DL Accuracy (%)': [
        results['dl_20']['accuracy'] * 100,
        results['dl_10']['accuracy'] * 100,
        results['dl_5']['accuracy'] * 100
    ],
    'FL Accuracy (%)': [
        results['fl_20']['accuracy'] * 100,
        results['fl_10']['accuracy'] * 100,
        results['fl_5']['accuracy'] * 100
    ]
})

comparison_df['Accuracy Difference (DL - FL)'] = comparison_df['DL Accuracy (%)'] - comparison_df['FL Accuracy (%)']

print("\n" + "="*80)
print("FL vs DL COMPARISON RESULTS")
print("="*80)
print(comparison_df.to_string(index=False))
print("="*80)

## 9. Visualize Comparison

In [ ]:
# Create comparison plots
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('FL vs DL Model Comparison', fontsize=16, fontweight='bold')

configurations = [(20, 'dl_20', 'fl_20'), (10, 'dl_10', 'fl_10'), (5, 'dl_5', 'fl_5')]

for idx, (config, dl_key, fl_key) in enumerate(configurations):
    # Training curve
    ax1 = axes[0, idx]
    dl_hist = results[dl_key]['history']
    fl_hist = results[fl_key]['history']
    
    # DL training curve
    ax1.plot(range(1, config + 1), dl_hist.history['accuracy'], 
             marker='o', label='DL - Training', linewidth=2, markersize=6)
    ax1.plot(range(1, config + 1), dl_hist.history['val_accuracy'], 
             marker='s', label='DL - Validation', linewidth=2, markersize=6)
    
    # FL training curve
    ax1.plot(fl_hist['round'], fl_hist['test_accuracy'], 
             marker='^', label='FL - Test', linewidth=2, markersize=6)
    
    ax1.set_title(f'{config} Rounds/Epochs - Accuracy', fontweight='bold')
    ax1.set_xlabel('Round/Epoch')
    ax1.set_ylabel('Accuracy')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Bar comparison
    ax2 = axes[1, idx]
    dl_acc = results[dl_key]['accuracy'] * 100
    fl_acc = results[fl_key]['accuracy'] * 100
    
    bars = ax2.bar(['DL', 'FL'], [dl_acc, fl_acc], 
                   color=['#2196F3', '#4CAF50'], alpha=0.8, width=0.6)
    
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.2f}%', ha='center', va='bottom', fontweight='bold')
    
    ax2.set_title(f'{config} Rounds/Epochs - Final Accuracy', fontweight='bold')
    ax2.set_ylabel('Test Accuracy (%)')
    ax2.set_ylim([0, 100])
    ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('fl_dl_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ Comparison plot saved as 'fl_dl_comparison.png'")

## 10. Detailed Performance Analysis

In [ ]:
# Get detailed predictions for each model
from sklearn.metrics import precision_score, recall_score, f1_score

print("\n" + "="*80)
print("DETAILED PERFORMANCE METRICS")
print("="*80 + "\n")

for config, dl_key, fl_key in configurations:
    print(f"\n{'='*40}")
    print(f"{config} Rounds/Epochs Configuration")
    print(f"{'='*40}")
    
    # DL predictions
    dl_pred = results[dl_key]['model'].predict(X_test_scaled, verbose=0)
    dl_pred_classes = np.argmax(dl_pred, axis=1)
    y_test_classes = np.argmax(y_test_cat, axis=1)
    
    # FL predictions
    fl_pred = results[fl_key]['model'].predict(X_test_scaled, verbose=0)
    fl_pred_classes = np.argmax(fl_pred, axis=1)
    
    print(f"\nDeep Learning Model:")
    print(f"  Accuracy:  {accuracy_score(y_test_classes, dl_pred_classes)*100:.2f}%")
    print(f"  Precision: {precision_score(y_test_classes, dl_pred_classes, average='weighted')*100:.2f}%")
    print(f"  Recall:    {recall_score(y_test_classes, dl_pred_classes, average='weighted')*100:.2f}%")
    print(f"  F1-Score:  {f1_score(y_test_classes, dl_pred_classes, average='weighted')*100:.2f}%")
    
    print(f"\nFederated Learning Model:")
    print(f"  Accuracy:  {accuracy_score(y_test_classes, fl_pred_classes)*100:.2f}%")
    print(f"  Precision: {precision_score(y_test_classes, fl_pred_classes, average='weighted')*100:.2f}%")
    print(f"  Recall:    {recall_score(y_test_classes, fl_pred_classes, average='weighted')*100:.2f}%")
    print(f"  F1-Score:  {f1_score(y_test_classes, fl_pred_classes, average='weighted')*100:.2f}%")
    
    print(f"\nPerformance Gap (DL - FL):")
    gap = accuracy_score(y_test_classes, dl_pred_classes) - accuracy_score(y_test_classes, fl_pred_classes)
    print(f"  {gap*100:+.2f}% accuracy difference")

## 11. Save Results Summary

In [ ]:
# Save comparison results to JSON
results_summary = {
    'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'architecture': '512-256-128-64',
    'dataset': 'UCI HAR',
    'configurations': []
}

for config, dl_key, fl_key in configurations:
    results_summary['configurations'].append({
        'rounds_epochs': config,
        'dl_accuracy': float(results[dl_key]['accuracy']),
        'fl_accuracy': float(results[fl_key]['accuracy']),
        'accuracy_gap': float(results[dl_key]['accuracy'] - results[fl_key]['accuracy'])
    })

# Save to JSON
with open('fl_dl_comparison_results.json', 'w') as f:
    json.dump(results_summary, f, indent=2)

# Save DataFrame to CSV
comparison_df.to_csv('fl_dl_comparison_results.csv', index=False)

print("\n✅ Results saved to:")
print("  - fl_dl_comparison_results.json")
print("  - fl_dl_comparison_results.csv")
print("  - fl_dl_comparison.png")

## 12. Summary and Conclusions

This notebook compared Federated Learning (FL) and Deep Learning (DL) models with identical architectures across three configurations:

### Key Findings:
- Both models use the same architecture (512-256-128-64)
- Training configurations: 20, 10, and 5 rounds/epochs
- FL uses subject-based data splitting to simulate federated clients
- Results show the trade-off between privacy (FL) and performance (DL)

### Expected Patterns:
1. **DL typically achieves higher accuracy** due to centralized training on all data
2. **FL maintains competitive accuracy** while preserving data privacy
3. **Performance gap decreases** with more training rounds/epochs
4. **FL benefits from federated averaging** across diverse clients

Use these results for your thesis comparison and analysis!

## 13. Training Time and Accuracy Comparison

Comparison of actual training times and accuracies from experiments.

In [ ]:
# Experimental results data
iterations = [3, 5, 10, 20]

# Training times in seconds
dl_times = [12, 14, 17, 25]
fl_times = [32, 54, 95, 188]  # 32s, 54s, 1m35s=95s, 3m8s=188s

# Accuracies
dl_accuracies = [90.33, 92.64, 92.16, 94.33]
fl_accuracies = [50.49, 66.03, 81.71, 89.58]

# Create figure with 2 subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('FL vs DL: Training Time and Accuracy Comparison', fontsize=16, fontweight='bold')

# Plot 1: Training Time Comparison
ax1.plot(iterations, dl_times, marker='o', linewidth=2.5, markersize=10, 
         label='DL', color='#2196F3', linestyle='-')
ax1.plot(iterations, fl_times, marker='s', linewidth=2.5, markersize=10, 
         label='FL', color='#4CAF50', linestyle='-')

# Add value labels on points for training time
for i, (it, dl_t, fl_t) in enumerate(zip(iterations, dl_times, fl_times)):
    ax1.text(it, dl_t, f'{dl_t}s', ha='center', va='bottom', fontweight='bold', fontsize=10)
    ax1.text(it, fl_t, f'{fl_t}s', ha='center', va='bottom', fontweight='bold', fontsize=10)

ax1.set_xlabel('Number of Iterations', fontsize=12, fontweight='bold')
ax1.set_ylabel('Training Time (seconds)', fontsize=12, fontweight='bold')
ax1.set_title('Training Time Comparison', fontsize=14, fontweight='bold')
ax1.legend(fontsize=12)
ax1.grid(True, alpha=0.3)
ax1.set_xticks(iterations)

# Plot 2: Accuracy Comparison
ax2.plot(iterations, dl_accuracies, marker='o', linewidth=2.5, markersize=10, 
         label='DL', color='#2196F3', linestyle='-')
ax2.plot(iterations, fl_accuracies, marker='s', linewidth=2.5, markersize=10, 
         label='FL', color='#4CAF50', linestyle='-')

# Add value labels on points for accuracy
for i, (it, dl_a, fl_a) in enumerate(zip(iterations, dl_accuracies, fl_accuracies)):
    ax2.text(it, dl_a, f'{dl_a:.2f}%', ha='center', va='bottom', fontweight='bold', fontsize=10)
    ax2.text(it, fl_a, f'{fl_a:.2f}%', ha='center', va='bottom', fontweight='bold', fontsize=10)

ax2.set_xlabel('Number of Iterations', fontsize=12, fontweight='bold')
ax2.set_ylabel('Test Accuracy (%)', fontsize=12, fontweight='bold')
ax2.set_title('Accuracy Comparison', fontsize=14, fontweight='bold')
ax2.legend(fontsize=12)
ax2.grid(True, alpha=0.3)
ax2.set_xticks(iterations)
ax2.set_ylim([40, 100])

plt.tight_layout()
plt.savefig('fl_dl_time_accuracy_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ Training time and accuracy comparison plot saved as 'fl_dl_time_accuracy_comparison.png'")

In [ ]:
# Create detailed comparison table
comparison_table = pd.DataFrame({
    'Iterations': iterations,
    'DL Time (seconds)': dl_times,
    'FL Time (seconds)': fl_times,
    'Time Ratio (FL/DL)': [f'{fl/dl:.1f}x' for fl, dl in zip(fl_times, dl_times)],
    'DL Accuracy (%)': dl_accuracies,
    'FL Accuracy (%)': fl_accuracies,
    'Accuracy Gap (%)': [round(dl - fl, 2) for dl, fl in zip(dl_accuracies, fl_accuracies)]
})

print("\n" + "="*100)
print("EXPERIMENTAL RESULTS: FL vs DL COMPARISON")
print("="*100)
print(comparison_table.to_string(index=False))
print("="*100)
print("\nKey Observations:")
print(f"• FL is {fl_times[-1]/dl_times[-1]:.1f}x slower than DL on average (20 iterations)")
print(f"• DL achieves {dl_accuracies[-1]:.2f}% accuracy vs FL's {fl_accuracies[-1]:.2f}% ({dl_accuracies[-1]-fl_accuracies[-1]:.2f}% gap)")
print(f"• FL requires more iterations to achieve competitive accuracy")
print(f"• At 20 iterations, FL approaches DL performance with only {dl_accuracies[-1]-fl_accuracies[-1]:.2f}% accuracy difference")